# 05 - Base RAG Generation Evaluation

Runs Base RAG on the locked benchmark using the selected retrieval setup. Current retrieval decision: dense top-5 context, because dense retrieval produced the best article-level metrics and the tested reranker did not improve them.

In [ ]:
from pathlib import Path
import json
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print('Not running in Google Colab; using local filesystem paths.')

DRIVE_ROOT = Path('/content/drive/MyDrive/rag')
sys.path = [str(DRIVE_ROOT)] + [p for p in sys.path if p != str(DRIVE_ROOT)]

config = json.loads((DRIVE_ROOT / 'project_config.json').read_text(encoding='utf-8'))
benchmark_csv = DRIVE_ROOT / config['benchmark_csv']
index_root = DRIVE_ROOT / config.get('official_index_root', 'indexes/official_law_v3')
output_dir = DRIVE_ROOT / 'outputs/generation_eval'

for path in [benchmark_csv, index_root / 'index_manifest.json']:
    if not path.exists():
        raise FileNotFoundError(path)

benchmark_csv, index_root, output_dir

In [ ]:
import importlib.util
import subprocess
import sys

required_modules = {
    'transformers': 'transformers',
    'accelerate': 'accelerate',
    'bitsandbytes': 'bitsandbytes',
    'sentence_transformers': 'sentence-transformers',
    'faiss': 'faiss-cpu',
    'rank_bm25': 'rank-bm25',
    'tqdm': 'tqdm',
}

missing_packages = [package for module, package in required_modules.items() if importlib.util.find_spec(module) is None]
if missing_packages:
    print('Installing missing packages:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
else:
    print('All generation dependencies are already installed.')

If the configured LLM is gated on Hugging Face, set `HF_TOKEN` in Colab secrets or run `from huggingface_hub import login; login()` before the next cell. Use the same exact model later for LoRA/QLoRA fine-tuning.

In [ ]:
import torch
from src.generation import run_rag_generation

device = 'cuda' if torch.cuda.is_available() else 'cpu'
llm_model = config.get('base_llm_model', 'google/gemma-2-2b-it')

# Set limit=10 for a quick smoke test first. Then set limit=None for the full 190-question run.
limit = 10

run_config = run_rag_generation(
    benchmark_csv=benchmark_csv,
    index_root=index_root,
    output_predictions_csv=output_dir / 'base_rag_predictions_v1_smoke.csv',
    output_run_config_json=output_dir / 'base_rag_run_config_v1_smoke.json',
    llm_model=llm_model,
    retriever_mode='dense',
    top_k_context=5,
    candidate_k=30,
    dense_weight=1.0,
    bm25_weight=0.0,
    device=device,
    max_new_tokens=384,
    temperature=0.2,
    top_p=0.9,
    limit=limit,
    load_in_4bit=True,
)

run_config

In [ ]:
from src.evaluation_qa import evaluate_generation_predictions

summary = evaluate_generation_predictions(
    predictions_csv=output_dir / 'base_rag_predictions_v1_smoke.csv',
    output_eval_csv=output_dir / 'base_rag_eval_v1_smoke.csv',
    output_summary_json=output_dir / 'base_rag_eval_summary_v1_smoke.json',
)

summary

In [ ]:
import pandas as pd

preview = pd.read_csv(output_dir / 'base_rag_eval_v1_smoke.csv', dtype=str, keep_default_na=False)
preview[['question_id', 'question', 'gold_answer', 'generated_answer', 'retrieved_citations', 'token_f1', 'rouge_l', 'citation_present', 'citation_gold_match']].head(3)

After the smoke test looks sane, rerun the generation cell with `limit = None` and output filenames without `_smoke`:

- `base_rag_predictions_v1.csv`
- `base_rag_run_config_v1.json`
- `base_rag_eval_v1.csv`
- `base_rag_eval_summary_v1.json`